In [1]:
import numpy as np

# 模拟某策略的月度收益（示例数据，%）
rets = np.array([2.1, -1.3, 3.0, 0.5, -0.8, 2.4, 1.1, -2.0, 1.8, 0.9, 2.6, -0.4]) / 100

# 月收益均值与标准差
mu_m = rets.mean()          # 月均收益
std_m = rets.std(ddof=1)    # 月波动（样本标准差）

# 年化（12个月，假设复利近似）
mu_y  = (1 + mu_m)**12 - 1
std_y = std_m * np.sqrt(12)

# 夏普比率（无风险利率按年2%给，月0.02/12）
rf_m = 0.02 / 12
sharpe = (mu_m - rf_m) / std_m * np.sqrt(12)

print(f"月均收益={mu_m:.2%}  年化收益≈{mu_y:.2%}")
print(f"月波动={std_m:.2%}   年化波动≈{std_y:.2%}")
print(f"夏普比率≈{sharpe:.2f}")


月均收益=0.83%  年化收益≈10.36%
月波动=1.64%   年化波动≈5.70%
夏普比率≈1.39


In [ ]:
## Day2 分散化：为什么 "别把鸡蛋放一个篮子"

### 知识点 1・相关性与协方差

- **协方差**：两只资产收益**同向波动**的程度（正 = 同涨同跌，负 = 你涨我跌）；
- **相关系数 ρ（-1~1）**：协方差的 "标准化"——**ρ=1 完全同向，ρ=-1 完全反向，ρ=0 无关**；
- **一句话**：分散化能不能起作用，**全看 ρ 有多小**—— 这是本章唯一的 "元变量"。

### 知识点 2・组合方差公式（两只资产）

- **组合预期收益**：`E(Rp) = w1·E(R1) + w2·E(R2)`（加权平均，简单）；
- **组合方差**：`σp² = w1²σ1² + w2²σ2² + 2·w1·w2·ρ·σ1·σ2`（**关键在这**）；
- **核心结论（公式自己会说话）**：**只要 ρ<1，组合风险就低于 "加权平均风险"**——ρ 越小，组合风险越低，**ρ=-1 时理论上可以完全消除**；
- **注意**：分散化**不降低预期收益**（收益是加权平均），只降低风险 ——**这就是 "免费午餐" 的含义：同样的收益，更小的风险**。

### 知识点 3・分散化的本质（升级认知）

- 资产数量增加 → 组合风险下降，但**下降有极限**（市场 /systematic 风险消不掉，个股 /idiosyncratic 风险能消）——**这个 "消不掉的底"，就是后面 Day4 β 的由来**；
- **经理人 / 投资人版本**：**相关性低的资产搭配 = 风险共享，相关性高的资产堆一起 = 假分散**（2008 年所有资产一起跌，就是 "假分散" 的教科书）。

In [7]:
import numpy as np

# 两只资产的年波动与相关系数（示例数据）
sig1, sig2 = 0.25, 0.30      # 资产1、2的年波动
rho = 0.30                   # 相关系数（低相关→分散化有效）

# 不同权重组合的组合风险
for w1 in [0.0, 0.25, 0.5, 0.75, 1.0]:
    w2 = 1 - w1
    var_p = w1**2*sig1**2 + w2**2*sig2**2 + 2*w1*w2*rho*sig1*sig2
    std_p = np.sqrt(var_p)
    print(f"w1={w1:.2f}  w2={w2:.2f}  组合波动={std_p:.2%}")

# 观察：50/50 时组合波动应明显低于"加权平均0.275"
print("加权平均波动(参考)=", (0.5*0.25+0.5*0.30))


w1=0.00  w2=1.00  组合波动=30.00%
w1=0.25  w2=0.75  组合波动=25.09%
w1=0.50  w2=0.50  组合波动=22.22%
w1=0.75  w2=0.25  组合波动=22.19%
w1=1.00  w2=0.00  组合波动=25.00%
加权平均波动(参考)= 0.275


In [ ]:
### 知识点 1・马科维茨组合理论（一句话）

- **核心思想**：投资人不是只看收益，而是**在 "给定风险下追求最高收益"**—— 于是所有可能的组合里，有一排 "最优组合"；
- **前提假设**（知道就行，别纠结）：用均值代表收益、方差代表风险、资产收益服从正态 ——**现实不完美，但框架极其有用**。

### 知识点 2・有效前沿（Efficient Frontier）

- **定义**：**同样风险下收益最高、同样收益下风险最低**的那排组合连成的曲线；
- **曲线下方的组合**：同样的风险收益更差 → "无效组合"，**没人该选**；
- **曲线形状**：向左上方凸 ——**想多赚一点，风险要付出越来越多**（收益的 "边际代价" 递增）。

预览

查看代码

### 知识点 3・三个关键点（本章落点）

- **① 最小方差组合（GMV）**：有效前沿最左端 ——**风险最低**的组合，不管收益；
- **② 切点组合（Tangency Portfolio）**：CML 与有效前沿的切点 ——**每单位风险的回报最高**（夏普最高）——**引入无风险资产后，理性人全选它**；
- **③ 资本市场线（CML）**：无风险资产与切点组合的连线 ——**线上任意一点 = 无风险资产 + 切点组合的搭配**（借入杠杆可以 "线上右边"）；
- **对量化的意义**：**有效前沿告诉你 "最优解长什么样"**——Day14-15 你会亲手用 Python 把它画出来，并找到你自己的切点组合。

In [ ]:
# 阶段二 讲义・Day4–6 资产定价

**本阶段目标**：回答一个核心问题 ——**"风险" 到底值多少钱？** 从 CAPM 的 "一个风险因子" 到多因子的 "一套风险因子"，再到市场有效性的 "哪些钱能赚"。Day5 直接接上你的量化因子库。

---

## Day4 CAPM：风险怎么变成预期收益

### 知识点 1・系统风险 vs 个股风险（衔接 Day2 的 "消不掉的底"）

- **系统风险（市场风险，β 抓的）**：整个市场一起波动 ——**分散化消不掉**（2008 年全跌）；
- **个股风险（特有风险）**：单只股票自己的事 ——**分散化能消掉**；
- **CAPM 的核心命题**：市场只给**消不掉的系统风险**付钱 ——**个股风险不该有额外回报**（因为分散化免费消掉它）。

### 知识点 2・CAPM 公式与 β

- **公式**：`E(Ri) = Rf + βi × [E(Rm) − Rf]`
  - `Rf` 无风险利率（国债）；`E(Rm)−Rf` 市场风险溢价；`βi` 资产对市场的敏感度；
- **β 的含义**：**市场涨 1%，它涨 / 跌百分之几**——β=1 跟市场同步，β=1.5 放大 1.5 倍，β=0.5 温和，β=0 与市场无关（如现金）；
- **β 怎么算（你会算）**：**β = 资产收益与市场收益的协方差 ÷ 市场方差**—— 也就是 "资产收益对市场收益回归的斜率"（你的计量功底直接复用）。

### 知识点 3・SML 证券市场线（CAPM 的图形）

- **SML**：横轴 β、纵轴预期收益 ——**一条从 Rf 出发、斜率 = 市场风险溢价的直线**；
- **线上 = 定价合理**（β 多少就该赚多少）；**线上方 = 被低估（α>0，白捡）**；**线下方 = 被高估（α<0）**；
- **α（阿尔法）**：实际收益 − CAPM 该赚的收益 ——**量化里 "超额收益" 就是它**—— 你的策略如果能稳定 α>0，就说明在赚 "不该赚的钱"。

In [8]:
import numpy as np

# 示例：估算一只股票（或你的因子组合）的 β 和 α
# 用"资产超额收益 ~ 市场超额收益"回归（你的计量功底）
asset_ret = np.array([1.5, -0.8, 2.2, 0.4, -1.1, 1.9, 0.7, -1.6, 1.2, 0.5])  # 资产收益(%)
mkt_ret   = np.array([1.0, -0.5, 1.4, 0.3, -0.7, 1.2, 0.5, -1.0, 0.8, 0.3])  # 市场收益(%)
rf = 0.02 / 12  # 无风险利率（月）

# 超额收益
ae = asset_ret - rf*100
me = mkt_ret - rf*100

# β = cov(资产,市场)/var(市场)；α = 资产均值超额 − β*市场均值超额
beta = np.cov(ae, me, ddof=1)[0,1] / np.var(me, ddof=1)
alpha = ae.mean() - beta * me.mean()

print(f"β≈{beta:.2f}   月α≈{alpha:.3f}%（年化α≈{(1+alpha/100)**12-1:.1%}）")


β≈1.57   月α≈0.067%（年化α≈0.8%）


In [ ]:
- **预期输出（示意）**：β 接近 1.2~1.4（弹性略大）、α 若为正 → **该资产在 CAPM 眼里 "多赚了"**。

**对照实践**：把 `asset_ret` 换成你策略的收益、`mkt_ret` 换成指数收益 ——**算出你自己的 β 和 α，这就是你策略的 "定价体检报告"**。

In [ ]:
## Day5 多因子模型：从 "一个因子" 到 "一套因子"（量化重点）

### 知识点 1・为什么 CAPM 不够

- **实证发现**：小市值股票长期跑赢大市值、低估值（高账面市值比）股票跑赢高估值 ——**这些超额和 β 无关**；
- **结论**：市场风险不是唯一的定价因子 ——**"风险" 是多元的**。

### 知识点 2・Fama-French 三因子模型（框架 + 你会跑的回归）

- **公式**：`E(Ri) = Rf + βm·MKT + βs·SMB + βh·HML`
  - **MKT（市场因子）**：市场超额收益（CAPM 那个）；
  - **SMB（规模因子，Small Minus Big）**：小市值 − 大市值的收益差 ——**小盘溢价**；
  - **HML（价值因子，High Minus Low）**：高账面市值比 − 低账面市值比的收益差 ——**价值溢价**；
- **含义**：一只股票的预期收益 = 市场溢价 + 它的大小盘敞口 × 小盘溢价 + 它的价值敞口 × 价值溢价 ——**β 从 1 个变成 3 个**（后面还有五因子：再加盈利 RMW、投资 CMA）；
- **对量化的意义（直接可用）**：
  - **因子 = 可交易的策略**：SMB/HML 本身就是 "买入小盘卖大盘"" 买入价值卖成长 " 的**多空组合**——**这就是因子投资 / 聪明贝塔的源头**；
  - **归因**：你的策略收益 = 因子敞口收益 + α（真正的超额）——**把收益拆成 "吃因子" 和 "真本事" 两部分**；
  - **你之前在计量里学的多元回归，就是这里的主力工具**。

### 知识点 3・因子怎么构建（给你 "工厂" 的图纸）

- 步骤：**排序 → 分组 → 算组间收益差 → 得到因子序列**；
  - SMB：每月按市值排序，小盘组收益 − 大盘组收益；
  - HML：每月按账面市值比排序，高组 − 低组；
- **样本注意**：因子序列要够长（20 年 +）、市值加权、剔除极端值 ——**因子质量决定回归质量**。

In [9]:
import numpy as np

# 模拟数据（示意）：某策略月度超额收益 + 三个因子（MKT/SMB/HML），各60个月
np.random.seed(42)
n = 60
mkt = np.random.normal(0.005, 0.04, n)   # 市场因子
smb = np.random.normal(0.003, 0.03, n)   # 规模因子
hml = np.random.normal(0.002, 0.03, n)   # 价值因子
# 策略收益 = 0.9*MKT + 0.3*SMB + 0.2*HML + 真α(0.5%/月) + 噪声
alpha_true = 0.005
strat = 0.9*mkt + 0.3*smb + 0.2*hml + alpha_true + np.random.normal(0, 0.01, n)

# 三因子回归（用 statsmodels，你的计量老伙计）
import statsmodels.api as sm
X = sm.add_constant(np.column_stack([mkt, smb, hml]))
model = sm.OLS(strat, X).fit()
print(model.summary().tables[1])  # 看 intercept(=α)、mkt/smb/hml 的系数


                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0056      0.001      4.051      0.000       0.003       0.008
x1             0.9373      0.038     24.736      0.000       0.861       1.013
x2             0.3331      0.050      6.675      0.000       0.233       0.433
x3             0.1864      0.047      3.932      0.000       0.091       0.281


In [ ]:
## Day7 VaR 与 CVaR：给 "最坏情况" 量尺寸

### 知识点 1・VaR（Value at Risk，风险价值）

- **定义**：给定置信水平（如 95%）和时间（如 1 天），**最坏亏损的阈值**——"95% 的概率，明天亏损不会超过 X 元"；
- **三种算法（都要会）**：
  1. **历史模拟法**：把过去 N 天收益排序，取第 5% 分位 ——**简单、不假设分布，最常用**；
  2. **参数法（方差 - 协方差）**：假设正态，`VaR = −(μ + z·σ)·仓位`——**快，但正态假设在尾部不靠谱**；
  3. **蒙特卡洛**：模拟上万条路径，取分位 ——**灵活、能放复杂假设，但算得慢**；
- **关键短板（必须知道）**：**VaR 只告诉你 "阈值在哪"，不告诉你 "超过阈值后平均亏多少"**—— 两个分布尾部完全不同的组合可以有相同 VaR。

### 知识点 2・CVaR（条件风险价值 / Expected Shortfall）

- **定义**：**超过 VaR 阈值之后，平均亏损是多少**—— 补上 VaR 的短板；
- **一句话区别**：VaR 问 "最坏 5% 的边界在哪"，CVaR 问 "最坏 5% 的里面，平均有多惨"——**风控和监管现在越来越爱 CVaR（尾部更诚实）**。

In [10]:
import numpy as np

# 示例：组合日收益（%），算 95% VaR 和 CVaR（历史模拟法）
np.random.seed(7)
rets = np.random.normal(0.0005, 0.015, 500) * 100   # 500天日收益（%）

var95 = np.percentile(rets, 5)                       # 历史法 VaR（5%分位）
cvar95 = rets[rets <= var95].mean()                  # CVaR：尾部平均

print(f"95% VaR（1天）≈ {var95:.2f}%   （95%概率亏损不超过此值）")
print(f"95% CVaR（1天）≈ {cvar95:.2f}% （一旦进入最坏5%，平均亏这么多）")


95% VaR（1天）≈ -2.48%   （95%概率亏损不超过此值）
95% CVaR（1天）≈ -3.12% （一旦进入最坏5%，平均亏这么多）


In [ ]:
- **预期输出（示意）**：VaR≈−2.4%、CVaR≈−3.1%——**CVaR 更吓人，也更真实**。

**对照实践**：把收益序列换成你的策略日收益，重跑 ——**每份策略报告都该带上这两个数**（Day15 会用）。

In [ ]:
## Day8 波动率建模：风险是会变的

### 知识点 1・三种波动率（从简单到进阶）

1. **历史波动率**：过去 N 天的标准差 ——**简单但迟钝**（对 "最近变大了" 反应慢）；
2. **EWMA（指数加权）**：给近期收益更大权重 ——**对市场变化更敏感**（RiskMetrics 的标配，λ≈0.94）；
3. **GARCH（广义自回归条件异方差）**：波动率自己也 "波动聚集"——**大波动后面跟着大波动**（金融市场的著名特征）——GARCH 建模 "波动率的均值回归"；

- **记忆钩子**：历史 = 平均、EWMA = 偏近期、GARCH = 会聚散 ——**越往后越懂 "波动率不是常数"**。

### 知识点 2・波动率聚集与厚尾（为什么要建模）

- **波动率聚集**：平静期和风暴期交替 ——**用单一标准差预测未来，风暴期会严重低估风险**；
- **厚尾（肥尾）**：真实收益的极端值比正态分布预测的多（"黑天鹅" 比正态说的频繁）——**这是 VaR 参数法（正态假设）的致命伤**；
- **实务结论**：**风控里 "最坏情况" 永远按厚尾想，不按正态想**——**这正是 CVaR + 压力测试要一起用的原因**。

### 知识点 3・波动率微笑（期权视角，给你个 "彩蛋"）

- **现象**：期权隐含波动率随行权价呈 "微笑" 曲线（两端高、中间低）——**市场自己在给 "极端行情" 加价**；
- **含义**：市场早就知道**尾部比正态重**——**微笑曲线就是市场对 "厚尾" 的定价**；
- **对你的意义**：做期权或看衍生品时，**别用单一波动率，看整条微笑曲线**（衔接你之前读的《期权期货及其他衍生品》）。

In [11]:
import numpy as np

# 示例：日收益，比较 历史波动率 vs EWMA 波动率
np.random.seed(1)
rets = np.random.normal(0, 0.02, 250)   # 250天日收益（假设波动率恒定，示意）

hist_sig = np.std(rets, ddof=1)          # 历史波动

# EWMA：波动率 = sqrt((1-λ)·Σ λ^(k-1)·r²)
lam = 0.94
var_ewma = 0.0
for r in rets:
    var_ewma = lam * var_ewma + (1 - lam) * r**2
ewma_sig = np.sqrt(var_ewma)

print(f"历史波动={hist_sig:.3%}   EWMA波动≈{ewma_sig:.3%}")
# 真实场景里，把序列换成长短波动交替的数据，能明显看到 EWMA 跟得快


历史波动=1.908%   EWMA波动≈2.087%


In [ ]:
- **预期输出（示意）**：两者接近（数据是恒定波动）——**换成 "平静→风暴" 的真实数据，EWMA 会显著更快抬升**。

**对照实践**：构造一段 "前半年平静、后半年暴动" 的收益序列（后半段 σ 拉大 3 倍），重跑对比 ——**亲眼看到 EWMA 更快报警**

In [ ]:
## Day9 风险预算、压力测试、情景分析

### 知识点 1・风险预算（Risk Budgeting）

- **定义**：把 "总风险"（组合波动 / VaR）**按比例分给各个资产 / 策略**—— 每个部门或策略分到多少 "风险额度"；
- **和资金配置的区别（关键）**：**钱是平均分的，风险不是**—— 高波动策略分更少钱但占更多风险额度 ——**按风险配，比按钱配更科学**；
- **对你的应用**：多策略组合（动量 + 价值 + 套利）——**每加一个策略，先算它贡献多少风险，再决定仓位**。

### 知识点 2・压力测试（Stress Test）

- **定义**：**人为设定极端场景，看组合亏多少**——"如果市场一天跌 8% 怎么办"" 如果利率跳 200bp 怎么办 "；
- **与 VaR 的区别**：VaR 说 "95% 概率亏多少"，**压力测试问 "那 5% 里，最狠的场景亏多少"**——**专治 VaR 的盲区**；
- **历史场景库（可复用）**：2008 金融危机、2020 疫情熔断、2022 加息周期 ——**"过去发生过的极端，未来可能重演"**。

### 知识点 3・情景分析的完整动作（本章落点）

- **三步走**：
  1. **选情景**：历史重演（2008）+ 假想极端（-20% 单日）+ 业务特写（加息 / 汇率跳贬）；
  2. **算影响**：组合重定价 / 重算收益 ——**每类资产在情景下怎么动**；
  3. **定对策**：**亏损能否承受？需要减仓 / 对冲 / 加保证金吗？**—— 压力测试的目的不是 "吓自己"，是**提前决定 "出事时怎么办"**；
- **一句话**：**风险管理的终点不是算出数字，是 "数字背后有个预案"**。

In [12]:
import numpy as np

# 压力测试（示意逻辑）：假设 2008 式场景，各类资产同时大跌
scenario = {"股票": -0.45, "债券": +0.05, "商品": -0.30, "现金": 0.0}
weights  = {"股票": 0.6, "债券": 0.25, "商品": 0.1, "现金": 0.05}
loss = sum(weights[k] * scenario[k] for k in weights)
print(f"2008情景下组合损失≈{loss:.1%}")   # 全代码思路：重定价→汇总→对照承受力


2008情景下组合损失≈-28.8%


In [ ]:
**预期输出（示意）**：组合损失≈−29%——**接着问：扛得住吗？要不要把股票降到 40%？**

In [ ]:
# 阶段四 讲义・Day10–13 组合管理

**本阶段目标**：把前三阶段的理论**落地成 "怎么配钱"**—— 从大类配置到再平衡、从固收久期到另类投资。四天走完，你脑子里要有 "一套完整组合怎么搭、怎么管、怎么归因"。

---

## Day10 资产配置：组合的 "总设计师"

### 知识点 1・战略配置 vs 战术配置

- **战略资产配置（SAA）**：**长期、宏观**—— 根据你的目标 / 风险承受力，定 "股 / 债 / 现金 / 商品" 的基准比例（如 60/40）——**组合 90% 以上的收益波动由它决定**；
- **战术资产配置（TAA）**：**短期、相机**—— 在基准上小幅偏离（如 "近期看多股票，暂时 65/35"）——**锦上添花，不是主食**；
- **一句话**：**SAA 决定你 "是什么组合"，TAA 决定你 "这季度微调多少"—— 先定 SAA，再谈 TAA**。

### 知识点 2・配置的逻辑（衔接 Day2-3）

- **股债低相关**是组合的 "压舱石"：股票给增长、债券给缓冲 ——**两类低相关资产搭配，有效前沿往左上方走**；
- **配比不是拍脑袋**：按**风险承受力（能承受多少回撤）× 时间跨度（多久要用钱）× 目标收益**倒推；
- **生命周期视角**：年轻（时间长、能扛波动）→ 股票多；临近用钱 → 债券多 ——**配置是 "个人参数" 的函数**。

### 知识点 3・配置的落地指标

- **盯两个数**：组合**波动率（σ）**和**最大回撤（Max Drawdown）**—— 配完先问 "最坏跌多少，我还睡得着吗"；
- **工具思维**：核心 — 卫星（Core-Satellite）——**核心**是指数 / 被动（拿 β），**卫星**是主动 / 因子（找 α）—— 和你 Day6 的 "β 打底 + α 加餐" 完全一致。

### 对照实践（可立即做）

- 写下你的**个人参数**（能承受最大回撤、资金可用年限、目标年化），设计一个 SAA 基准比例，再写一条 TAA 偏离规则（如 "沪深 300 市盈率 < 12 时股票 + 5%"）。

---

## Day11 再平衡与绩效归因：管好组合的 "日常"

### 知识点 1・再平衡（Rebalancing）

- **为什么必须做**：股票涨多了，仓位自动变重 ——**组合会漂移出你的风险设定**（想配 60/40，涨完变成 75/25，风险超标）；
- **两种做法**：
  1. **日历再平衡**：每季度 / 每年固定调回目标比例 ——**简单、纪律化**；
  2. **阈值再平衡**：偏离超过 ±5% 才调 ——**减少交易成本，容忍小漂移**；
- **心理考验（本书式提醒）**：再平衡 =**"卖涨买跌"**—— 涨得好的卖掉、跌的买进，**逆人性，但长期是收益来源之一**（低买高卖制度化）。

### 知识点 2・绩效归因（Performance Attribution）

- **问题**：这个季度组合跑赢 / 跑输基准 ——**功劳 / 锅是配置的还是选股的？**（衔接 Day5 归因思路，从 "因子" 扩到 "配置层"）；
- **两层拆解**：
  - **配置效应**：大类比例偏离基准带来的超额（"多配了股票，股票又涨了"）；
  - **选股效应**：同类资产内部选得比基准好带来的超额（"选的那只白酒跑赢行业"）；
- **对你的意义**：**归因让你知道 "超额从哪来"—— 找不到来源的超额，下次就不一定还在**（衔接 Day6"说得清才算数"）。

### 知识点 3・组合监控仪表（给你的 "仪表盘"）

表格

| 指标 | 频率 | 用途 |
| --- | --- | --- |
| 组合波动 / 回撤 | 周 | 风险是否越界 |
| 与基准的超额（α） | 月 | 超额是否稳定 |
| 配置偏离度 | 季 | 触发再平衡 |
| 换手 / 成本 | 季 | 交易是不是太频繁 |

### 对照实践（可立即做）

- 给 Day10 的 SAA 定**再平衡规则**（日历 or 阈值，写清参数）；给组合定**3 个监控指标**和各自阈值。

---

## Day12 固收组合管理：久期与凸性（衔接《固定收益证券》）

### 知识点 1・久期（Duration）：价格的 "利率敏感度"

- **定义**：利率变动 1%，债券价格变动百分之几（近似）——**久期是债券的 "β"**；
- **直观**：久期 5 年 → 利率升 1%，价格约跌 5%；**久期越长，对利率越敏感**（久期 7 vs 3，谁怕加息一目了然）；
- **组合应用**：**组合久期 = 各债券久期的市值加权**——"我想让组合抗加息，就降组合久期"（买短债 / 浮息债）。

### 知识点 2・凸性（Convexity）：久期的 "修正项"

- **问题**：久期是线性近似（切线），**但价格 — 收益率关系是曲线（凸的）**—— 利率大动时，久期会低估价格变化；
- **凸性补丁**：利率大跌时债券涨得比久期预测的多，利率大涨时跌得比预测的少 ——**凸性是好事（涨多跌少）**；
- **一句话**：**久期告诉你方向，凸性告诉你 "曲线弯了多少"—— 久期 + 凸性才是完整的价格预测**。

预览

查看代码

### 知识点 3・固收组合的两个实操动作

- **免疫（Immunization）**：让组合久期≈负债久期 ——**利率怎么动，资产和负债同涨同跌，净值不受伤**（养老金 / 保险的标配）；
- **骑乘曲线 / 杠铃策略**：集中短端 + 长端（杠铃）、或集中中间（子弹）——**不同久期结构 = 不同利率赌注**；
- **对你的意义**：**久期是 "固收版的 β"，凸性是 "固收版的期权价值"**—— 管债和管股，思维方式是通的。

In [13]:
# 久期近似（简化）：P≈面值100、票息5%、YTM4%
def price_approx(dur, y, dy, conv=0):
    # 价格变动 ≈ −久期×Δy + 0.5×凸性×(Δy)²
    return -dur * dy + 0.5 * conv * dy**2
print(price_approx(7, 0.04, 0.01))   # 久期7：利率+1% → 价格约−7%


-0.07


In [14]:
import numpy as np
import matplotlib
matplotlib.use("Agg")                       # 无显示环境
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC"]  # 中文显示
plt.rcParams["axes.unicode_minus"] = False

# ---- 1. 数据：4只资产，模拟月收益（真实场景可换成指数/个股历史）----
np.random.seed(2026)
n_assets, n_months = 4, 120
mu = np.array([0.012, 0.008, 0.006, 0.010])   # 月均收益（示意）
cov = np.random.rand(n_assets, n_assets) * 0.1
cov = (cov + cov.T) / 2                        # 对称
np.fill_diagonal(cov, [0.04, 0.03, 0.02, 0.035])  # 方差（示意）
rets = np.random.multivariate_normal(mu, cov, n_months)

# ---- 2. 权重扫描画有效前沿 ----
N_SIM = 20000
np.random.seed(7)
w = np.random.dirichlet(np.ones(n_assets), N_SIM)   # 随机权重(和为1,非负)

port_mu  = w @ mu
port_var = np.einsum("ij,jk,ik->i", w, cov, w)      # w·Σ·w'
port_sig = np.sqrt(port_var)

# 上包络线：固定波动区间内取收益最高点（简化画法）
from scipy.optimize import minimize
def obj(ww): return ww @ cov @ ww                   # 目标：最小方差
def ret_con(ww): return ww @ mu
cons = [{"type": "eq", "fun": lambda ww: np.sum(ww) - 1}]
bnds = [(0, 1)] * n_assets
x0 = np.ones(n_assets) / n_assets

frontier = []
for target in np.linspace(mu.min(), mu.max(), 40):
    cons_ret = cons + [{"type": "eq", "fun": lambda ww, t=target: ww @ mu - t}]
    res = minimize(obj, x0, bounds=bnds, constraints=cons_ret)
    if res.success:
        frontier.append((res.x @ mu, np.sqrt(res.fun)))

# GMV：全局最小方差
res_gmv = minimize(obj, x0, bounds=bnds, constraints=cons)
gmv_mu, gmv_sig = res_gmv.x @ mu, np.sqrt(res_gmv.fun)

# ---- 3. 画图 ----
plt.figure(figsize=(7, 5.5))
plt.scatter(port_sig, port_mu, s=2, c="lightblue", label="随机组合")
fx = np.array([p[1] for p in frontier]); fy = np.array([p[0] for p in frontier])
plt.plot(fx, fy, "r-", lw=2, label="有效前沿")
plt.scatter([gmv_sig], [gmv_mu], c="green", s=80, marker="*", label=f"GMV 最小方差组合")
plt.xlabel("年化波动率(近似月σ×√12)"); plt.ylabel("月均收益")
plt.title("有效前沿与最小方差组合（模拟数据 · 示意）")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("efficient_frontier.png", dpi=150)
print("已生成 efficient_frontier.png")
print(f"GMV组合：月收益={gmv_mu:.3%}  波动≈{gmv_sig:.2%}  权重={np.round(res_gmv.x,2)}")


C:\Users\lenovo\AppData\Local\Temp\ipykernel_41264\3720272159.py:15: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  rets = np.random.multivariate_normal(mu, cov, n_months)
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans CJK SC
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans CJK SC
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans CJK SC
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans CJK SC
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans CJK SC
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans CJK SC
findfont: Generic family 'sans-serif' not found because none of the following families were found: Noto Sans C

已生成 efficient_frontier.png
GMV组合：月收益=0.600%  波动≈14.14%  权重=[0. 0. 1. 0.]


In [15]:
import numpy as np

# ---- 1. 组合收益序列（用Day14的组合权重+资产收益算）----
np.random.seed(99)
n = 500
asset_rets = np.random.normal([0.001, 0.0008, 0.0006, 0.001],
                              [0.02, 0.015, 0.01, 0.018], size=(n, 4))
w_opt = np.array([0.25, 0.20, 0.30, 0.25])     # 沿用Day14得到的权重（示例）
port_ret = asset_rets @ w_opt                    # 组合日收益

# ---- 2. VaR / CVaR（历史模拟法，日口径）----
var95 = np.percentile(port_ret, 5)
cvar95 = port_ret[port_ret <= var95].mean()

# ---- 3. 压力测试（示意情景）----
shock = {"温和熊市": -0.10, "金融危机": -0.30, "极端黑天鹅": -0.50}
for name, s in shock.items():
    print(f"{name}: 组合单日损失≈{s:.0%}  → 应对：{( '减仓/对冲' if s <= -0.2 else '持有观察' )}")

print(f"95% VaR(日)≈{var95:.2%}   CVaR(日)≈{cvar95:.2%}")
print(f"年化波动≈{np.std(port_ret,ddof=1)*np.sqrt(250):.1%}  夏普≈{(port_ret.mean()/np.std(port_ret,ddof=1))*np.sqrt(250):.2f}")


温和熊市: 组合单日损失≈-10%  → 应对：持有观察
金融危机: 组合单日损失≈-30%  → 应对：减仓/对冲
极端黑天鹅: 组合单日损失≈-50%  → 应对：减仓/对冲
95% VaR(日)≈-1.18%   CVaR(日)≈-1.44%
年化波动≈12.6%  夏普≈2.52
